
# QUANTIZATION POUR FINE-TUNING LLM

---

##  Le problème des gros modèles

Un LLM comme :

* Meta (LLaMA 3 70B)
* Google (Gemma)
* OpenAI

contient **des milliards de paramètres (weights)**.

Ces poids sont stockés en **FP32 (32 bits)**.
-   👉 C’est très précis.
-   👉 Mais ça prend énormément de mémoire.

Ton GPU :

* a une VRAM limitée
* coûte cher
* ne peut pas charger 200+ GB de modèle

---

## L’idée de la quantization

On va faire quelque chose de simple :

> Garder l’intelligence du modèle,
> mais réduire la taille des nombres.

Exemple :

Avant :

```
7.2345678912
```

Après :

```
7.23
```

On passe de :

* 32 bits → 16 bits (FP16)
* 32 bits → 8 bits (INT8)

-   👉 Moins de mémoire
-   👉 Inference plus rapide
-   👉 Possible sur mobile / edge device

**Convert higher memory format → lower memory format**

---

## Pourquoi c’est crucial ?

Parce que ça permet :

* Déploiement mobile
* Edge devices
* Réduction des coûts cloud
* Accélération inference

---

## Le compromis

Quand on compresse :

* On perd un peu d’information
* Donc parfois un peu d’accuracy

---

##  Calibration (version simple)

Calibration =
👉 Trouver comment “squeezer” les nombres proprement.

On convertit :

```
0 – 1000
```

vers :

```
0 – 255
```

C’est juste une mise à l’échelle intelligente.

Pas besoin d’entrer trop dans les équations en cours si public non math.

---

##  Deux modes de quantization

### A) Post Training Quantization (PTQ)


Pipeline :

Pretrained model
→ Calibration
→ Quantized model

Simple.
Rapide.
Mais petite perte d’accuracy possible.

---

### B) Quantization Aware Training (QAT)

Pipeline :

Trained model
→ Quantization
→ Fine-tuning
→ Quantized model

Ici :

* Le modèle apprend avec la compression
* Il s’adapte
* Moins de perte

👉 C’est ce qu’on utilise en fine-tuning moderne.

---


> Quantization est la fondation de toutes les techniques modernes comme LoRA et QLoRA.


# Ce que ça change réellement dans un LLM
---

Un LLM est essentiellement une énorme matrice de poids.

En FP32 :

* 1 poids = 4 bytes
* 7B paramètres ≈ 28GB
* 70B paramètres ≈ 280GB

👉 Impossible sur GPU standard.

En INT8 :

* 1 poids = 1 byte
* 4× moins de mémoire

En 4-bit :

* 8× moins de mémoire

---

### Symmetric vs Asymmetric (intuition claire)

#### 🔹 Symmetric

* Distribution centrée autour de 0
* Plus simple
* Souvent utilisé pour weights

#### 🔹 Asymmetric

* Distribution décalée
* Nécessite “zero point”
* Plus flexible
* Souvent pour activations


---

### PTQ vs QAT (impact pratique)

| PTQ                        | QAT                              |
| -------------------------- | -------------------------------- |
| Compression après training | Compression intégrée au training |
| Rapide                     | Plus stable                      |
| Peut perdre en précision   | Meilleure robustesse             |
| Utilisé pour inference     | Utilisé pour fine-tuning         |

---

### Pourquoi c’est clé pour Fine-Tuning moderne

Fine-tuning full model en FP32 :

-   ❌ Trop coûteux
-   ❌ Trop de VRAM

Quantization permet :

* Charger modèle 4-bit
* Garder poids gelés
* Adapter seulement petites matrices (LoRA)

👉 C’est exactement la logique derrière QLoRA.